# Huấn luyện Mô hình Đánh giá IELTS bằng 1-LoRA (Multi-Task Fine-Tuning với Unsloth)

Notebook này thực hiện **Giai đoạn 3: Huấn luyện 1-LoRA** cho dự án **Automated Essay Scoring (AES)**.
Chúng ta chuyển đổi từ kiến trúc 4 LoRA adapters riêng biệt trong `study_5` sang **một LoRA adapter duy nhất (1-LoRA)** chấm điểm đồng thời cả 4 tiêu chí (TR, CC, LR, GRA) và trả về định dạng JSON thống nhất.

**Cấu hình Cache & Tránh rác ổ C:**
- Sử dụng tệp `.env` để cấu hình đường dẫn cache Hugging Face và Unsloth trên ổ `T:`. Toàn bộ dữ liệu mô hình tải về sẽ không lưu ở ổ C.

**Tối ưu hóa phần cứng (RTX 4060 8GB VRAM):**
- Sử dụng thư viện **Unsloth** để tăng tốc độ huấn luyện gấp 2 lần và giảm 60% mức tiêu thụ VRAM.
- Lượng hóa 4-bit (`load_in_4bit=True`).
- Bộ tối ưu hóa `paged_adamw_8bit` giải phóng bộ nhớ khi đạt đỉnh.
- Gradient Checkpointing giúp tiết kiệm bộ nhớ KV Cache.

In [ ]:
import os
from dotenv import load_dotenv

# Nạp các biến môi trường từ tệp .env ở gốc dự án trước khi import Unsloth hoặc Transformers
load_dotenv(os.path.abspath("../.env"))

import sys
import json
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Thêm đường dẫn src/ để sử dụng rag_utils
sys.path.append(os.path.abspath("../src"))
from rag import rag_utils

## 1. Thiết lập Cấu hình & Tải Mô hình Llama-3.1-8B Lượng hóa 4-bit

In [ ]:
max_seq_length = 2048 # Đủ chứa: New Essay (500 tokens) + 2 Reference Essays (1000 tokens) + Prompt & Output (500 tokens)
dtype = None           # Tự động phát hiện (Float16 hoặc Bfloat16)
load_in_4bit = True    # Bắt buộc bật để tiết kiệm VRAM trên card 8GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 2. Thiết lập Cấu hình LoRA (PEFT)

Cấu hình các tham số rank `r=16` và `lora_alpha=32` để đảm bảo adapter học tốt các tác vụ đa mục tiêu mà không bị quá khớp.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, # Tối ưu hóa bằng 0 trong Unsloth
    bias = "none",    # Tối ưu hóa none
    use_gradient_checkpointing = True, # Giảm thiểu tối đa VRAM tiêu thụ
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 3. Thiết kế Prompt Template & Tiền xử lý dữ liệu

### Tránh nghẽn RAG khi Train bằng phương pháp Pre-computation:
Thay vì truy vấn cơ sở dữ liệu Vector động trong quá trình train (làm giảm tốc độ train đi 10 lần), chúng ta thực hiện truy xuất RAG một lần cho tất cả mẫu Train/Val và lưu vào DataFrame tạm.

In [ ]:
# Load Vector DB để truy xuất trước
VECTOR_DB_DIR = "../data/processed/chroma_db"
vectordb = rag_utils.load_vector_db(VECTOR_DB_DIR)

# Đọc tập dữ liệu sạch
df_train = pd.read_csv("../data/processed/train.csv")
df_val = pd.read_csv("../data/processed/val.csv")

def add_rag_context_column(df):
    contexts = []
    for idx, row in df.iterrows():
        # Tìm 2 ví dụ tham khảo
        docs = rag_utils.retrieve_examples(vectordb, row['essay'], k=2)
        context_str = rag_utils.format_rag_context(docs)
        contexts.append(context_str)
    df['rag_context'] = contexts
    return df

print("Đang truy xuất ngữ cảnh RAG cho tập Train...")
df_train = add_rag_context_column(df_train)
print("Đang truy xuất ngữ cảnh RAG cho tập Val...")
df_val = add_rag_context_column(df_val)

### Thiết kế cấu trúc Prompt & Sinh JSON đầu ra

Chúng ta định nghĩa hàm mapping để biến đổi từng dòng dữ liệu thành định dạng cấu trúc Prompt của Llama 3.1.

In [ ]:
IELTS_EVAL_PROMPT_TEMPLATE = """You are a highly experienced IELTS writing examiner. Your goal is to provide a precise and consistent evaluation of an essay by following a structured reasoning process.

**CONTEXT (Reference Essays with Scores):**
{context}

**NEW ESSAY TO GRADE:**
{question}

**EVALUATION PROCESS (Think step-by-step):**
1. Task Response (TR) Analysis: Assess how well the 'NEW ESSAY' addresses the prompt. Compare its quality to the TR scores in the 'CONTEXT'.
2. Coherence and Cohesion (CC) Analysis: Assess structure, paragraphing, and linking. Compare to CC scores in the 'CONTEXT'.
3. Lexical Resource (LR) Analysis: Assess range and accuracy of vocabulary. Compare to LR scores in the 'CONTEXT'.
4. Grammatical Range and Accuracy (GRA) Analysis: Assess grammar range and accuracy. Compare to GRA scores in the 'CONTEXT'.

**FINAL OUTPUT FORMAT (Strict JSON):**
Your entire response MUST be a single valid JSON object containing exactly these fields. Do NOT include markdown code blocks or explanations outside JSON.
{{
  "Task_Response": {{
    "Band": {tr_band},
    "Comment": "{tr_comment}"
  }},
  "Coherence_and_Cohesion": {{
    "Band": {cc_band},
    "Comment": "{cc_comment}"
  }},
  "Lexical_Resource": {{
    "Band": {lr_band},
    "Mistakes": {lr_mistakes},
    "Corrections": {lr_corrections},
    "Comment": "{lr_comment}"
  }},
  "Grammatical_Range_and_Accuracy": {{
    "Band": {gra_band},
    "Mistakes": {gra_mistakes},
    "Corrections": {gra_corrections},
    "Comment": "{gra_comment}"
  }},
  "General_Feedback": "{general_feedback}"
}}

JSON Response:
"""

In [ ]:
EOS_TOKEN = tokenizer.eos_token

def clean_json_string(text):
    if pd.isna(text):
        return ""
    # Escape dấu ngoặc kép để tránh hỏng chuỗi JSON
    return str(text).replace('"', '\"').replace('\n', ' ').strip()

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    essays = examples["essay"]
    rag_contexts = examples["rag_context"]
    
    tr_bands = examples["TR_Band"]
    tr_comments = [clean_json_string(c) for c in examples["TR_Comment"]]
    
    cc_bands = examples["CC_Band"]
    cc_comments = [clean_json_string(c) for c in examples["CC_Comment"]]
    
    lr_bands = examples["LR_Band"]
    lr_mistakes = [str(m) if str(m).startswith('[') else '[]' for m in examples["LR_Mistakes"]]
    lr_corrections = [str(c) if str(c).startswith('[') else '[]' for c in examples["LR_Corrections"]]
    lr_comments = [clean_json_string(c) for c in examples["LR_Comment"]]
    
    gra_bands = examples["GRA_Band"]
    gra_mistakes = [str(m) if str(m).startswith('[') else '[]' for m in examples["GRA_Mistakes"]]
    gra_corrections = [str(c) if str(c).startswith('[') else '[]' for c in examples["GRA_Corrections"]]
    gra_comments = [clean_json_string(c) for c in examples["GRA_Comment"]]
    
    general_feedbacks = [clean_json_string(c) for c in examples["General_Feedback"]]
    
    texts = []
    for i in range(len(prompts)):
        question_str = f"Prompt: {prompts[i]}\nEssay: {essays[i]}"
        
        # Ghép nhãn đầu ra JSON
        response_json = (
            f'{{\n'
            f'  "Task_Response": {{\n'
            f'    "Band": {float(tr_bands[i])},\n'
            f'    "Comment": "{tr_comments[i]}"\n'
            f'  }},\n'
            f'  "Coherence_and_Cohesion": {{\n'
            f'    "Band": {float(cc_bands[i])},\n'
            f'    "Comment": "{cc_comments[i]}"\n'
            f'  }},\n'
            f'  "Lexical_Resource": {{\n'
            f'    "Band": {float(lr_bands[i])},\n'
            f'    "Mistakes": {lr_mistakes[i]},\n'
            f'    "Corrections": {lr_corrections[i]},\n'
            f'    "Comment": "{lr_comments[i]}"\n'
            f'  }},\n'
            f'  "Grammatical_Range_and_Accuracy": {{\n'
            f'    "Band": {float(gra_bands[i])},\n'
            f'    "Mistakes": {gra_mistakes[i]},\n'
            f'    "Corrections": {gra_corrections[i]},\n'
            f'    "Comment": "{gra_comments[i]}"\n'
            f'  }},\n'
            f'  "General_Feedback": "{general_feedbacks[i]}"\n'
            f'}}'
        )
        
        # Trộn prompt hệ thống
        text = IELTS_EVAL_PROMPT_TEMPLATE.format(
            context=rag_contexts[i],
            question=question_str,
            tr_band=float(tr_bands[i]), tr_comment=tr_comments[i],
            cc_band=float(cc_bands[i]), cc_comment=cc_comments[i],
            lr_band=float(lr_bands[i]), lr_mistakes=lr_mistakes[i], lr_corrections=lr_corrections[i], lr_comment=lr_comments[i],
            gra_band=float(gra_bands[i]), gra_mistakes=gra_mistakes[i], gra_corrections=gra_corrections[i], gra_comment=gra_comments[i],
            general_feedback=general_feedbacks[i]
        ) + response_json + EOS_TOKEN
        
        texts.append(text)
    return {"text": texts}

# Chuyển sang định dạng HuggingFace Dataset
train_dataset = Dataset.from_pandas(df_train).map(formatting_prompts_func, batched=True)
val_dataset = Dataset.from_pandas(df_val).map(formatting_prompts_func, batched=True)
print("✔ Dữ liệu huấn luyện đã sẵn sàng!")

## 4. Thiết lập Huấn luyện viên (SFTTrainer)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,      # Tối ưu hóa cực hạn VRAM
        gradient_accumulation_steps = 16,     # Tích lũy gradient mô phỏng batch size = 16
        warmup_steps = 10,
        num_train_epochs = 3,                 # Huấn luyện đầy đủ 3 epochs
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),       # RTX 4060 hỗ trợ bf16 (rất mượt mà)
        logging_steps = 1,
        optim = "paged_adamw_8bit",           # Tiết kiệm bộ nhớ tối đa
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "../checkpoints",        # Thư mục lưu checkpoint sau mỗi epoch
        evaluation_strategy = "epoch",
        save_strategy = "epoch",
        report_to = "none",
    ),
)

## 5. Bắt đầu Huấn luyện & Lưu Adapter

In [ ]:
print("Đang huấn luyện mô hình...")
trainer_stats = trainer.train()

# Lưu LoRA Adapter cuối cùng
ADAPTER_DIR = "../adapters/llama_8b_1lora_aes"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✔ Đã lưu adapter tại: {ADAPTER_DIR}")